In [2]:
# =============================================================================
# Sleep-EDF Expanded — Cassette → 10 sharded NPZ
# -----------------------------------------------------------------------------
# X channels (order):
#   0: EEG Fpz-Cz     (100 Hz)
#   1: EEG Pz-Oz      (100 Hz)
#   2: EOG horizontal (100 Hz)
#   3: EMG submental  (upsampled from 1 Hz via zero-order hold)
#   4: subject_id     (constant; integer 0..82)
#
# Wake trimming policy per recording:
#   - Drop all excluded epochs (?, M, Movement time).
#   - Find first and last sleep epochs (N1/N2/N3/REM).
#   - Keep all sleep epochs between them.
#   - Keep up to 60 W epochs immediately before first sleep epoch.
#   - Keep up to 60 W epochs immediately after last sleep epoch.
#   - Drop all other W (WASO and distant Wake).
#
# Output:
#   processed/cassette_shards/_global_meta.npz
#   processed/cassette_shards/shard_00.npz
#   ...
#   processed/cassette_shards/shard_09.npz
#
# Each shard_XX.npz contains:
#   X : (n, 5, 3000)  float32
#   y : (n,)          int8    0=W, 1=N1, 2=N2, 3=N3, 4=REM
# =============================================================================

from __future__ import annotations

import gc
import re
import time
import warnings
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
import mne

warnings.filterwarnings("ignore", message=".*highpass filters.*")
warnings.filterwarnings("ignore", message=".*lowpass filters.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff.*")


# =============================================================================
# 1) CONFIGURATION
# =============================================================================

DATA_ROOT: Path     = Path(r"D:\Hackaton\sleep-edf-database-expanded-1.0.0")
CASSETTE_DIR: Path  = DATA_ROOT / "sleep-cassette"
SUBJECTS_XLS: Path  = DATA_ROOT / "SC-subjects.xls"
PROCESSED_DIR: Path = DATA_ROOT / "processed"
SHARD_DIR: Path     = PROCESSED_DIR / "cassette_shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

N_SHARDS: int     = 10
RANDOM_SEED: int  = 42

# --- Signal channels from EDF (in this exact order) -------------------------
SIGNAL_CHANNELS: tuple[str, ...] = (
    "EEG Fpz-Cz",
    "EEG Pz-Oz",
    "EOG horizontal",
    "EMG submental",
)

# --- Derived channels added to X --------------------------------------------
DERIVED_CHANNELS: tuple[str, ...] = ("subject_id",)

# Full channel order for X
X_CHANNEL_NAMES: tuple[str, ...] = SIGNAL_CHANNELS + DERIVED_CHANNELS   # length 5
N_X_CHANNELS: int = len(X_CHANNEL_NAMES)  # 5

X_DTYPE: type = np.float32
COMPRESS_NPZ: bool = False

EEG_SAMPLING_HZ: float    = 100.0
EPOCH_DURATION_SEC: float = 30.0
SAMPLES_PER_EPOCH: int    = int(EEG_SAMPLING_HZ * EPOCH_DURATION_SEC)  # 3000

WAKE_BUFFER_EPOCHS: int = 60   # 30 minutes of Wake before / after sleep

STAGE_MAP: dict[str, str] = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
}
EXCLUDED_STAGES: frozenset[str] = frozenset({
    "Sleep stage ?", "Sleep stage M", "Movement time",
})
STAGE_ORDER: tuple[str, ...] = ("W", "N1", "N2", "N3", "REM")
STAGE_TO_INT: dict[str, int] = {s: i for i, s in enumerate(STAGE_ORDER)}
SLEEP_LABELS: frozenset[str] = frozenset({"N1", "N2", "N3", "REM"})

# --- Only needed to map subject numbers from filenames ----------------------
KEY_RE = re.compile(r"^SC4(\d{2})(\d)[A-Z]0$")


# =============================================================================
# 2) EDF READING WITH ZERO-ORDER HOLD
# =============================================================================

def _native_rates_from_raw(raw: mne.io.BaseRaw) -> list[float]:
    extras = raw._raw_extras[0]
    n_samps = extras.get("n_samps")
    record_length = extras.get("record_length")
    if n_samps is None or record_length is None:
        return [float(raw.info["sfreq"])] * len(raw.ch_names)
    n_samps = np.asarray(n_samps, dtype=float)
    rec_len = float(np.max(np.asarray(record_length, dtype=float)))
    if rec_len <= 0:
        return [float(raw.info["sfreq"])] * len(raw.ch_names)
    rates = (n_samps / rec_len).tolist()
    if len(rates) != len(raw.ch_names):
        return [float(raw.info["sfreq"])] * len(raw.ch_names)
    return rates


def reconstruct_zoh(
    sig_100hz: np.ndarray,
    native_rate: float,
    target_rate: float = EEG_SAMPLING_HZ,
) -> np.ndarray:
    """Force zero-order-hold upsampling via np.repeat (never interpolation)."""
    ratio_f = target_rate / native_rate
    if not np.isclose(ratio_f, round(ratio_f)):
        raise ValueError(
            f"Non-integer ratio {ratio_f} ({native_rate} -> {target_rate})"
        )
    ratio = int(round(ratio_f))
    if ratio == 1:
        return sig_100hz.astype(X_DTYPE)
    n_full = len(sig_100hz) // ratio
    downsampled = sig_100hz[: n_full * ratio : ratio]
    upsampled = np.repeat(downsampled, ratio)
    if len(upsampled) < len(sig_100hz):
        pad = len(sig_100hz) - len(upsampled)
        upsampled = np.concatenate([upsampled, np.full(pad, downsampled[-1])])
    return upsampled[: len(sig_100hz)].astype(X_DTYPE)


def read_psg(path: Path) -> tuple[pd.DataFrame, list[str]]:
    """Read one PSG at 100 Hz with ZOH. Missing channels -> NaN."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        raw = mne.io.read_raw_edf(
            str(path), preload=True, stim_channel=None, verbose=False,
        )

    native_rates = _native_rates_from_raw(raw)
    rate_by_ch = dict(zip(raw.ch_names, native_rates))

    n_samples = int(round(raw.times[-1] * EEG_SAMPLING_HZ)) + 1
    n_samples = min(n_samples, raw.n_times)

    data: dict[str, np.ndarray] = {}
    missing: list[str] = []
    for name in SIGNAL_CHANNELS:
        if name not in raw.ch_names:
            missing.append(name)
            data[name] = np.full(n_samples, np.nan, dtype=X_DTYPE)
            continue
        idx = raw.ch_names.index(name)
        sig = raw.get_data(picks=[idx])[0]
        try:
            data[name] = reconstruct_zoh(sig, rate_by_ch[name])[:n_samples]
        except ValueError:
            data[name] = sig[:n_samples].astype(X_DTYPE)

    df = pd.DataFrame(data)
    raw.close()
    return df, missing


# =============================================================================
# 3) HYPNOGRAM
# =============================================================================

def find_hypnogram_for_psg(psg_path: Path) -> Path:
    key7 = psg_path.stem[:7]
    candidates = [
        p for p in psg_path.parent.glob("*-Hypnogram.edf")
        if p.stem[:7] == key7
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected 1 Hypnogram for {psg_path.name}, found "
            f"{len(candidates)}: {[c.name for c in candidates]}"
        )
    return candidates[0]


def read_hypnogram_stages(
    hypno_path: Path,
    n_epochs: int,
    epoch_sec: float = EPOCH_DURATION_SEC,
) -> np.ndarray:
    """Array of length n_epochs. Values: 'W','N1','N2','N3','REM', or np.nan."""
    ann = mne.read_annotations(str(hypno_path))
    labels = np.full(n_epochs, np.nan, dtype=object)
    for start, dur, desc in zip(ann.onset, ann.duration, ann.description):
        if desc in EXCLUDED_STAGES or desc not in STAGE_MAP:
            continue
        s = int(round(start / epoch_sec))
        e = min(int(round((start + dur) / epoch_sec)), n_epochs)
        if e > s:
            labels[s:e] = STAGE_MAP[desc]
    return labels


# =============================================================================
# 4) WAKE TRIMMING POLICY
# =============================================================================

def apply_wake_trimming(
    stages: np.ndarray,
    wake_buffer_epochs: int = WAKE_BUFFER_EPOCHS,
) -> np.ndarray:
    """
    Keep only:
      - All sleep epochs (N1, N2, N3, REM) between first and last sleep.
      - Up to `wake_buffer_epochs` W epochs immediately before first sleep.
      - Up to `wake_buffer_epochs` W epochs immediately after last sleep.
    Drop everything else: WASO, distant Wake, and NaN (excluded epochs).

    Returns a boolean mask of length n_epochs.
    """
    n = len(stages)
    keep = np.zeros(n, dtype=bool)
    if n == 0:
        return keep

    is_sleep = np.fromiter(
        (s in SLEEP_LABELS for s in stages), dtype=bool, count=n
    )
    sleep_idx = np.where(is_sleep)[0]
    if sleep_idx.size == 0:
        return keep  # no sleep -> drop entire recording

    first_sleep = int(sleep_idx[0])
    last_sleep  = int(sleep_idx[-1])

    # 1) Keep all sleep epochs between first and last sleep
    keep[first_sleep : last_sleep + 1] = is_sleep[first_sleep : last_sleep + 1]

    # 2) Keep up to N W epochs before first sleep (walk backwards)
    count = 0
    for i in range(first_sleep - 1, -1, -1):
        if count >= wake_buffer_epochs:
            break
        if stages[i] == "W":
            keep[i] = True
            count += 1

    # 3) Keep up to N W epochs after last sleep (walk forwards)
    count = 0
    for i in range(last_sleep + 1, n):
        if count >= wake_buffer_epochs:
            break
        if stages[i] == "W":
            keep[i] = True
            count += 1

    return keep


# =============================================================================
# 5) RECORDING KEY PARSING
# =============================================================================

def parse_recording_key(recording_key: str) -> tuple[int, int]:
    """From 'SC4001E0' -> (subject=0, night=1)."""
    m = KEY_RE.match(recording_key)
    if not m:
        raise ValueError(f"Unexpected recording key: {recording_key}")
    return int(m.group(1)), int(m.group(2))


# =============================================================================
# 6) SINGLE-RECORDING PROCESSING
# =============================================================================

def process_cassette_recording(
    psg_path: Path,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Returns (X_epochs, y_epochs, qc).
    X_epochs : (n_kept, 5, 3000) float32
    y_epochs : (n_kept,) int8
    """
    signal_df, missing = read_psg(psg_path)
    hyp_path = find_hypnogram_for_psg(psg_path)

    n_epochs = len(signal_df) // SAMPLES_PER_EPOCH
    usable = n_epochs * SAMPLES_PER_EPOCH
    signal_df = signal_df.iloc[:usable]

    stages = read_hypnogram_stages(hyp_path, n_epochs)

    # --- Wake trimming -------------------------------------------------------
    keep_epochs = apply_wake_trimming(stages)
    n_kept = int(keep_epochs.sum())

    recording_key = psg_path.stem.replace("-PSG", "")
    subject_id, night = parse_recording_key(recording_key)

    qc = {
        "recording_key": recording_key,
        "subject": subject_id,
        "night": night,
        "n_epochs_total": n_epochs,
        "n_epochs_kept": n_kept,
        "n_w_total": int((stages == "W").sum()),
        "n_w_kept": int(((stages == "W") & keep_epochs).sum()),
        "n_sleep_kept": int(
            np.isin(stages[keep_epochs], list(SLEEP_LABELS)).sum()
        ),
        "missing_channels": missing,
    }

    if n_kept == 0:
        empty_X = np.empty((0, N_X_CHANNELS, SAMPLES_PER_EPOCH), dtype=X_DTYPE)
        empty_y = np.empty((0,), dtype=np.int8)
        return empty_X, empty_y, qc

    # --- Build 4-channel signal array ---------------------------------------
    arr = signal_df[list(SIGNAL_CHANNELS)].to_numpy(dtype=X_DTYPE)
    arr = arr.reshape(n_epochs, SAMPLES_PER_EPOCH, len(SIGNAL_CHANNELS))
    arr = np.transpose(arr, (0, 2, 1)).astype(X_DTYPE, copy=False)

    X_signals = arr[keep_epochs]                     # (n_kept, 4, 3000)

    # --- Assemble 5-channel X -----------------------------------------------
    X = np.empty((n_kept, N_X_CHANNELS, SAMPLES_PER_EPOCH), dtype=X_DTYPE)
    X[:, :4, :] = X_signals                          # signals
    X[:, 4, :]  = float(subject_id)                  # subject_id

    # --- Labels --------------------------------------------------------------
    kept_stages = stages[keep_epochs]
    y = np.array([STAGE_TO_INT[str(s)] for s in kept_stages], dtype=np.int8)

    del signal_df, arr, X_signals

    return X, y, qc


def iter_cassette_psg(cassette_dir: Path) -> Iterator[Path]:
    for p in sorted(cassette_dir.glob("*-PSG.edf")):
        if p.stem.startswith("SC"):
            yield p


# =============================================================================
# 7) SHARD ASSIGNMENT
# =============================================================================

def assign_subjects_to_shards(
    psg_paths: list[Path],
    n_shards: int,
    seed: int,
) -> tuple[dict[int, int], list[list[Path]]]:
    rec_subj: dict[Path, int] = {}
    for psg in psg_paths:
        key = psg.stem.replace("-PSG", "")
        subj, _ = parse_recording_key(key)
        rec_subj[psg] = subj

    unique_subjects = np.array(sorted(set(rec_subj.values())))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_subjects)

    groups = np.array_split(unique_subjects, n_shards)
    subject_to_shard: dict[int, int] = {}
    for sid, group in enumerate(groups):
        for s in group.tolist():
            subject_to_shard[int(s)] = sid

    per_shard: list[list[Path]] = [[] for _ in range(n_shards)]
    for psg, subj in rec_subj.items():
        per_shard[subject_to_shard[subj]].append(psg)

    return subject_to_shard, per_shard


# =============================================================================
# 8) BULK PROCESSING
# =============================================================================

def process_all_to_shards(
    cassette_dir: Path,
    shard_dir: Path,
    n_shards: int = N_SHARDS,
    seed: int = RANDOM_SEED,
) -> tuple[list[dict], list[dict], dict[int, int]]:
    psg_paths = list(iter_cassette_psg(cassette_dir))
    print(f"Found {len(psg_paths)} Cassette PSG files")
    if not psg_paths:
        raise RuntimeError("No Cassette PSG files found.")

    subject_to_shard, per_shard = assign_subjects_to_shards(
        psg_paths, n_shards, seed
    )
    print(f"Assigned {len(subject_to_shard)} subjects into {n_shards} shards")
    for sid, recs in enumerate(per_shard):
        n_subj = len({parse_recording_key(p.stem.replace("-PSG", ""))[0]
                      for p in recs})
        print(f"  shard {sid:02d}: {n_subj:2d} subjects, {len(recs):2d} recordings")

    qc_rows: list[dict] = []
    issues: list[dict] = []
    t_start = time.time()
    bytes_per_epoch = N_X_CHANNELS * SAMPLES_PER_EPOCH * np.dtype(X_DTYPE).itemsize

    def _allocate(capacity: int) -> dict[str, np.ndarray]:
        return {
            "X": np.empty((capacity, N_X_CHANNELS, SAMPLES_PER_EPOCH), dtype=X_DTYPE),
            "y": np.empty((capacity,), dtype=np.int8),
        }

    def _grow(arrs: dict[str, np.ndarray], new_capacity: int, pos: int) -> dict[str, np.ndarray]:
        out: dict[str, np.ndarray] = {}
        for k, old in arrs.items():
            new = np.empty((new_capacity,) + old.shape[1:], dtype=old.dtype)
            new[:pos] = old[:pos]
            out[k] = new
        return out

    for shard_id, recs in enumerate(per_shard):
        if not recs:
            continue

        print(f"\n=== Shard {shard_id:02d} ({len(recs)} recordings) ===")

        # After wake trimming, expect ~700 epochs per recording on average.
        base_estimate = max(len(recs) * 700, 500)
        capacity = int(base_estimate * 1.15) + 25 * len(recs)
        print(f"  Initial capacity: {capacity:,} epochs "
              f"(~{capacity * bytes_per_epoch / 1e9:.2f} GB for X)")

        arrs = _allocate(capacity)
        pos = 0

        for psg in recs:
            try:
                X_ep, y_ep, qc = process_cassette_recording(psg)
            except Exception as exc:
                print(f"  ERROR [{psg.name}]: {type(exc).__name__}: {exc}")
                issues.append({
                    "recording_key": psg.stem,
                    "error": f"{type(exc).__name__}: {exc}",
                })
                continue

            n = len(y_ep)
            if n == 0:
                print(f"    - {qc['recording_key']:>10s}  (0 epochs after trimming)")
                qc_rows.append(qc)
                continue

            if pos + n > capacity:
                new_capacity = max(capacity * 2, pos + n + 500)
                print(f"  Growing shard arrays: {capacity:,} -> {new_capacity:,}")
                arrs = _grow(arrs, new_capacity, pos)
                capacity = new_capacity

            arrs["X"][pos:pos + n] = X_ep
            arrs["y"][pos:pos + n] = y_ep
            pos += n

            del X_ep, y_ep

            qc_rows.append({
                "recording_key": qc["recording_key"],
                "shard_id": shard_id,
                "subject": qc["subject"],
                "night": qc["night"],
                "n_epochs_total": qc["n_epochs_total"],
                "n_epochs_kept": qc["n_epochs_kept"],
                "n_w_total": qc["n_w_total"],
                "n_w_kept": qc["n_w_kept"],
                "n_sleep_kept": qc["n_sleep_kept"],
                "missing_channels": ",".join(qc["missing_channels"]),
            })

            print(
                f"    + {qc['recording_key']:>10s}  "
                f"kept={n:>4d} (W {qc['n_w_kept']:>3d}/{qc['n_w_total']:>4d})  "
                f"pos={pos:>7d}/{capacity:>7d}  "
                f"subject={qc['subject']:>2d}  night={qc['night']}"
            )

        if pos == 0:
            print("  (no epochs in this shard — skipping)")
            del arrs
            continue

        print(f"  Shard holds {pos:,} epochs (allocated {capacity:,})")
        shard_path = shard_dir / f"shard_{shard_id:02d}.npz"
        saver = np.savez_compressed if COMPRESS_NPZ else np.savez
        t0 = time.time()
        saver(
            shard_path,
            X=arrs["X"][:pos],
            y=arrs["y"][:pos],
        )
        dt = time.time() - t0
        size_gb = shard_path.stat().st_size / 1e9
        print(f"  Saved {shard_path.name}: {size_gb:.2f} GB in {dt:.1f} s")

        del arrs
        gc.collect()

    # --- Global meta (tiny) --------------------------------------------------
    np.savez(
        shard_dir / "_global_meta.npz",
        channel_names=np.asarray(X_CHANNEL_NAMES, dtype=str),
        label_names=np.asarray(STAGE_ORDER, dtype=str),
        sfreq=np.asarray(int(EEG_SAMPLING_HZ)),
        epoch_duration=np.asarray(int(EPOCH_DURATION_SEC)),
    )

    subj_shard_df = pd.DataFrame(
        sorted(subject_to_shard.items()),
        columns=["subject", "shard_id"],
    )
    subj_shard_df.to_csv(shard_dir / "subject_to_shard.csv", index=False)

    print(f"\nTotal elapsed: {time.time() - t_start:.1f} s")
    return qc_rows, issues, subject_to_shard


# =============================================================================
# 9) LOADING HELPERS
# =============================================================================

def load_shard(shard_path: Path) -> dict:
    with np.load(shard_path, allow_pickle=False) as d:
        return {k: d[k] for k in d.files}


def load_global_meta(shard_dir: Path) -> dict:
    with np.load(shard_dir / "_global_meta.npz", allow_pickle=False) as d:
        return {k: d[k] for k in d.files}


def iter_shards(shard_dir: Path) -> Iterator[tuple[str, dict]]:
    for p in sorted(shard_dir.glob("shard_*.npz")):
        yield p.stem, load_shard(p)


def iter_batches(
    shard_dir: Path,
    batch_size: int = 256,
    shuffle: bool = False,
    rng: np.random.Generator | None = None,
) -> Iterator[dict]:
    """Reads one shard at a time. Yields dicts with keys: X, y."""
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)

    shard_paths = sorted(shard_dir.glob("shard_*.npz"))
    if shuffle:
        rng.shuffle(shard_paths)

    for sp in shard_paths:
        data = load_shard(sp)
        n = len(data["y"])
        idx = np.arange(n)
        if shuffle:
            rng.shuffle(idx)

        for start in range(0, n, batch_size):
            sel = idx[start : start + batch_size]
            yield {k: v[sel] for k, v in data.items()}

        del data
        gc.collect()


# =============================================================================
# 10) MAIN
# =============================================================================

if __name__ == "__main__":
    qc_rows, issues, subject_to_shard = process_all_to_shards(
        CASSETTE_DIR, SHARD_DIR,
        n_shards=N_SHARDS, seed=RANDOM_SEED,
    )
    qc_df = (
        pd.DataFrame(qc_rows)
        .sort_values(["shard_id", "subject", "night"])
        .reset_index(drop=True)
    )
    qc_df.to_csv(PROCESSED_DIR / "cassette_manifest.csv", index=False)

    # =========================================================================
    # FINAL REPORT
    # =========================================================================
    print("\n" + "=" * 72)
    print("FINAL REPORT")
    print("=" * 72)

    print(f"Recordings processed         : {len(qc_df)}")
    print(f"Total epochs before trimming : {int(qc_df['n_epochs_total'].sum()):,}")
    print(f"Total epochs after trimming  : {int(qc_df['n_epochs_kept'].sum()):,}")
    print(f"Total W before trimming      : {int(qc_df['n_w_total'].sum()):,}")
    print(f"Total W kept                 : {int(qc_df['n_w_kept'].sum()):,}")
    print(f"W removed by trimming        : "
          f"{int(qc_df['n_w_total'].sum() - qc_df['n_w_kept'].sum()):,}")

    # Streaming class distribution
    class_counts = np.zeros(len(STAGE_ORDER), dtype=np.int64)
    n_total = 0
    for _, data in iter_shards(SHARD_DIR):
        class_counts += np.bincount(data["y"], minlength=len(STAGE_ORDER))
        n_total += len(data["y"])

    print("\nFinal class distribution (after trimming):")
    for i, s in enumerate(STAGE_ORDER):
        print(f"  {s:>4}: {int(class_counts[i]):>10,}  "
              f"({100 * class_counts[i] / max(n_total, 1):5.2f}%)")

    print("\nPer-recording summary (first 15 rows):")
    print(
        qc_df[["recording_key", "shard_id", "subject", "night",
               "n_epochs_total", "n_epochs_kept",
               "n_w_total", "n_w_kept", "n_sleep_kept"]]
        .head(15).to_string(index=False)
    )

    print("\nRecordings with missing channels:")
    miss = qc_df[qc_df["missing_channels"] != ""]
    if miss.empty:
        print("  (none)")
    else:
        for _, r in miss.iterrows():
            print(f"  {r['recording_key']}: {r['missing_channels']}")

    print("\nIssues (if any):")
    if not issues:
        print("  (none)")
    else:
        for it in issues:
            print(f"  {it}")

    print("\nSubject counts per shard:")
    print(qc_df.groupby("shard_id")["subject"].nunique().to_string())

Found 153 Cassette PSG files
Assigned 78 subjects into 10 shards
  shard 00:  8 subjects, 16 recordings
  shard 01:  8 subjects, 16 recordings
  shard 02:  8 subjects, 16 recordings
  shard 03:  8 subjects, 16 recordings
  shard 04:  8 subjects, 16 recordings
  shard 05:  8 subjects, 15 recordings
  shard 06:  8 subjects, 16 recordings
  shard 07:  8 subjects, 16 recordings
  shard 08:  7 subjects, 14 recordings
  shard 09:  7 subjects, 12 recordings

=== Shard 00 (16 recordings) ===
  Initial capacity: 13,279 epochs (~0.80 GB for X)
    +   SC4001E0  kept= 773 (W 120/1997)  pos=    773/  13279  subject= 0  night=1
    +   SC4002E0  kept=1064 (W 120/1885)  pos=   1837/  13279  subject= 0  night=2
    +   SC4251E0  kept= 959 (W 120/1921)  pos=   2796/  13279  subject=25  night=1
    +   SC4252E0  kept= 930 (W 120/1855)  pos=   3726/  13279  subject=25  night=2
    +   SC4281G0  kept=1054 (W 120/1854)  pos=   4780/  13279  subject=28  night=1
    +   SC4282G0  kept= 986 (W 120/1948)  pos